# Posterior sampling of rupture rates: full workflow

Samples the posterior of every rupture's annual rate in an NZSHM22 rupture set from the inversion's constraints, with an exact Gibbs sampler (`tools/gibbs.py`). Edit the configuration cell, then run all cells; everything is written to `RUN_DIR`, by default a new subfolder of `DATA_DIR` named after the settings. The notebook calls the scripts in `tools/`; its plumbing is in `tools/notebook_steps.py`, the settings that rarely change in `tools/run_config.py`. Requirements: numpy, scipy, pandas, matplotlib and a C compiler. What each step does and why: `docs/REPORT.md` ("The notebook, step by step", and the sections each setting names); what to check after a run: `docs/TESTING.md`.

In [ ]:
# Configuration: edit this cell, then run all cells in order. Each setting is explained in docs/REPORT.md.
import os, sys

DATA_DIR = '~/inversions/NZSHM22_Matrices-630k/'  # the input data's folder (only read); RUN_DIR is by default a new subfolder of it, named after the settings
A_FILE, D_FILE = 'cru_A.txt', 'cru_D.txt'         # the input files, in DATA_DIR
MISFIT_SLIP = 'linear'                            # how each data group is scored: slip rates 'linear' (split normals), 'log10' or 'half' (REPORT: MISFIT settings)
MISFIT_MFD = 'half'                               # MFD bins 'linear', 'log10' or 'half' (MFD_GR replaces it with each bin's own PDF)
MISFIT_PALEO = 'years'                            # paleo sites 'years' (their recurrence-interval PDFs), 'rate' (Gaussian) or 'half'
GROUP_WEIGHTS = None                             # 'equal': the slip rates, MFD bins and paleo rates weighted to count equally (a third of their effective measurements each); None: as they come
MFD_MONOTONIC = False                             # True: no bin's total above its lower neighbour's where the target does not rise (a hard constraint)
MFD_GR = False                                    # True: each bin scored by its own target PDF from NZSHM22's (b, N) uncertainty (tools/mfd_gr.py; REPORT: The MFD from N5 and b)
MFD_GR_B = (0.823, 0.959, 1.089)                  # with MFD_GR: b's 10th percentile, mean and 90th percentile (NZSHM22 upper plate)
MFD_GR_N5 = (2.65, 3.40, 4.57)                    # with MFD_GR: the mean N(M>=5) per year at each of those b-values (correlated)
MFD_GR_NSCALE = (0.66, 1.41)                      # with MFD_GR: the N scaling's 2-sigma values about 1
RECONSIDER_MAGUNCERT = True                       # with MFD_GR: a further factor on N for the catalogue's magnitude uncertainties, three equally weighted branches; or three numbers
MFD_GR_TAU = 0.025                                # with MFD_GR: the bins scored jointly about each draw's G-R curve (b and the level drawn under the priors above), each step within this many decades of the curve's (0.025: about 6%); None: each bin by its own PDF
SLIP_ALONG_RUPTURE = 'uniform'                    # each rupture's slip along it: 'uniform' (the inputs') or 'tapered' (UCERF3's sqrt-sine)
CFM_TIMEFRAME = True                              # True: the CFM's slip rates corrected for their timeframes (Coffey et al.)
CFM_TEMPLATE_RHO = 1                              # the CFM's template-rate faults' correlation (1 by default); None: independent
SLIP_RATE_FACTOR = 0.8                            # the fraction of each fault's slip rate the ruptures take up (0.8)
SLIP_SUM = True                                   # True: the summed slip rates along each cross-section, a soft term
SLIP_SUM_L0 = 100                                 # SLIP_SUM's weight: km of plate boundary per measurement (100)
PLATE_RATE = True                                 # True: each strip's share of the plate rate at most SLIP_RATE_FACTOR (a hard constraint)
RUN_MINUTES = 240                                 # the whole notebook's length in minutes; the run lengths are planned to fit (tools/run_plan.py; 240 by default)
TARGETS = 'production'                            # the convergence verdict's targets: 'exploratory', 'production' (default) or 'precise'
DISPERSED_STARTS = True                           # True (default): each chain from its own over-dispersed start (tools/dispersed_starts.py); False: all from the best fit
FLOOR = -10                                       # log10 of spike-and-slab's slab floor and the diagnostics' cutoff (-10)
REPRESENTATIVES = 100                             # weighted representative draws for hazard (100); 0: none
THREADS = 4                                       # threads for the sampler (the M2's performance cores: 4)
SEED = None                                       # None (default): from the start time; an integer: repeat a run exactly; 'legacy': 101, 102, ...

if '_NOTEBOOK_DIR' not in globals():       # this notebook's folder, remembered so this cell can be re-run after chdir
    _NOTEBOOK_DIR = os.path.abspath('.')
sys.path.insert(0, os.path.join(_NOTEBOOK_DIR, 'tools'))   # the scripts, in tools/ beside this notebook
from notebook_steps import (setup, sh, show, results_dir, rupture_dir, ref_args, cleaning_args, after_cleaning,
                            starts_args, plan_runs, gibbs_args, replicate_seed, runs_done, clear_solution, write_timings)
setup(globals())                           # checks the settings, derives the rest (tools/run_config.py: the fixed settings,
                                           # RUN_NAME, RUN_DIR), makes RUN_DIR, starts its log, finds the rupture set


## 1. Clean and characterise

Cleans the inputs into the problem the sampler sees (`cru_clean.npz`), with the slip rates rebuilt from the CFM and, with `MFD_GR`, the bins' PDFs; finds the best nonnegative fit (`nnls/`) and the near-duplicate ruptures; prints the fit check. Diagnostics in `prelim_summary.txt`. On request, from the command line: the cleaning figure (`preliminaries.py --figure`) and the targets old and new (`plot_targets.py`).

In [ ]:
sh('preliminaries.py', *cleaning_args())        # the run's settings as preliminaries.py takes them
FIT = after_cleaning()                           # an earlier cleaning's outputs removed; the problem summarized


**The best fit's figures** (`nnls/fig_*_nnls.png`): the best nonnegative fit against the data, drawn as the posterior's are below.

In [ ]:
_rup = ['--ruptures', rupture_dir()] if rupture_dir() else []   # the fit figures in physical units, with the rupture set
_sets = ['nnls/nnls_vectors.npz:best fit (NNLS)']
for _script, _fig, _more in (('plot_slip_rates.py', 'nnls/fig_slip_rates_nnls.png', _rup),   # the best fit's fit to the data
                             ('plot_rate_magnitude.py', 'nnls/fig_rate_magnitude_nnls.png', ['--bins-block', BIN_BLOCK, f'--diag-floor={FLOOR}']),
                             ('plot_paleo_rates.py', 'nnls/fig_paleo_rates_nnls.png', ['--bins-block', BIN_BLOCK] + _rup)):
    sh(_script, *_sets, '--problem', 'cru_clean.npz', '--out', _fig, '--slip-block', SLIP_BLOCK, *_more, *ref_args())
    show(_fig)

## 2. Sampling

Every chain starts from its own over-dispersed start (`DISPERSED_STARTS`: half randomized best fits, half prior draws, each made feasible; `tools/dispersed_starts.py`); the run lengths are planned to fit `RUN_MINUTES` from two short calibration runs (`tools/run_plan.py`, printed and saved as `run_plan.json`); then `gibbs.py` samples spike-and-slab with every term and constraint the settings ask for. The runs go in `spikeslab/`.

In [ ]:
if DISPERSED_STARTS:                               # one start per chain, over-dispersed
    sh('dispersed_starts.py', *starts_args())
PLAN, P = plan_runs()                              # calibration and the run lengths (or a test's RUN_PLAN)
MAIN = []
for r in range(P['replicates']):
    sh('gibbs.py', *gibbs_args(f'main_rep{r}', replicate_seed(r), P))
    MAIN.append(os.path.join(RESULTS, f'main_rep{r}'))
runs_done()


In [ ]:
#MAIN = RUNS = [os.path.join(RESULTS, f'main_rep{r}') for r in range(1)]

## 3. Convergence

`check_convergence.py` gives the verdict against `TARGETS` (`PASS` or `CHECK`, naming the targets missed) and the run plan's predicted effective draws beside those achieved (`convergence_report.txt`); `block_fit.py` each data type's fit (`block_fit_report.txt`).

In [ ]:
OUT = results_dir()                                 # reports, tables and figures go in spikeslab/
sh('check_convergence.py', *MAIN, '--problem', 'cru_clean.npz', '--targets', TARGETS, '--plan', 'run_plan.json', f'--diag-floor={FLOOR}',
   '--out', os.path.join(OUT, 'convergence_report.txt'))
sh('block_fit.py', *MAIN, '--problem', 'cru_clean.npz', '--slip-block', SLIP_BLOCK, '--bins-block', BIN_BLOCK,
   '--out', os.path.join(OUT, 'block_fit_report.txt'))     # each row block's fit

## 4. Results

`make_tables.py`: per rupture, subsection and bin, the posterior's rates and exceedance probabilities, and the rate vectors (`rate_vectors.csv`, `.npz`; `tables_report.txt`).

In [ ]:
sh('make_tables.py', *MAIN, '--problem', 'cru_clean.npz', '--outdir', OUT, '--slip-block', SLIP_BLOCK,
   '--bins-block', BIN_BLOCK)                       # tables, rate vectors and tables_report.txt, in spikeslab/

**Representative draws for hazard** (`REPRESENTATIVES`): weighted actual draws, columns `rep_01`, ... of `rate_vectors.csv` (`select_report.txt`).

In [ ]:
if REPRESENTATIVES > 0 and os.path.exists(os.path.join(OUT, 'rate_vectors.npz')):   # weighted representative draws
    sh('select_draws.py', *MAIN, '--problem', 'cru_clean.npz', '--vectors', os.path.join(OUT, 'rate_vectors'),
       '--k', REPRESENTATIVES, '--slip-block', SLIP_BLOCK, '--bins-block', BIN_BLOCK)
else:
    print('representative draws skipped' + (' (REPRESENTATIVES = 0)' if REPRESENTATIVES <= 0 else ': no rate vectors'))

**Rates by magnitude** (`fig_rate_magnitude.png`).

In [ ]:
vec = os.path.join(results_dir(), 'rate_vectors.npz')
if os.path.exists(vec):                              # rates by magnitude, from the rate vectors' mean column
    fig = os.path.join(results_dir(), 'fig_rate_magnitude.png')
    sh('plot_rate_magnitude.py', vec, '--problem', 'cru_clean.npz', '--out', fig, '--bins-block', BIN_BLOCK,
       '--slip-block', SLIP_BLOCK, f'--diag-floor={FLOOR}', *ref_args())
    show(fig)
else:
    print('rates-by-magnitude figure skipped: it needs the rate vectors, written by the results cell above')

**Slip rates** (`fig_slip_rates.png`).

In [ ]:
vec = os.path.join(results_dir(), 'rate_vectors.npz')
if os.path.exists(vec):                              # slip-rate targets against the mean model's predictions
    fig = os.path.join(results_dir(), 'fig_slip_rates.png')
    sh('plot_slip_rates.py', vec, '--problem', 'cru_clean.npz', '--out', fig, '--slip-block', SLIP_BLOCK, *ref_args(),
       *(['--ruptures', rupture_dir()] if rupture_dir() else []))     # slip rates in mm/yr, with the rupture set
    show(fig)
else:
    print('slip-rate figure skipped: it needs the rate vectors, written by the results cell above')

**Paleo rates** (`fig_paleo_rates.png`).

In [ ]:
vec = os.path.join(results_dir(), 'rate_vectors.npz')
if os.path.exists(vec):                              # paleo-rate targets against the model's predictions
    fig = os.path.join(results_dir(), 'fig_paleo_rates.png')
    sh('plot_paleo_rates.py', vec, '--problem', 'cru_clean.npz', '--out', fig, '--slip-block', SLIP_BLOCK,
       '--bins-block', BIN_BLOCK, *ref_args(),
       *(['--ruptures', rupture_dir()] if rupture_dir() else []))     # rates per year and each site's ranges, with the rupture set
    show(fig)
else:
    print('paleo-rate figure skipped: it needs the rate vectors, written by the results cell above')

**Faults one by one** (`fig_slip_sections*.png`) **and how the rates vary together** (`corr_faults.png`, `corr_bins.png`; `covariance_report.txt`).

In [ ]:
vec = os.path.join(results_dir(), 'rate_vectors.npz')
if os.path.exists(vec):                              # the named faults one by one, and their ruptures
    fig = os.path.join(results_dir(), 'fig_slip_sections.png')
    _rd = rupture_dir()                              # its named_faults.json names the faults
    sh('plot_slip_sections.py', vec, '--problem', 'cru_clean.npz', '--out', fig, '--slip-block', SLIP_BLOCK,
       '--bins-block', BIN_BLOCK, *(['--ruptures', _rd] if _rd else []), *ref_args())
    for _f in ('', '_ruptures_1', '_ruptures_2', '_ruptures_by_magnitude_1', '_ruptures_by_magnitude_2'):
        if os.path.exists(fig.replace('.png', _f + '.png')):   # the rupture figures come in two parts
            show(fig.replace('.png', _f + '.png'))
    if all(os.path.exists(r + '_kept.npy') for r in MAIN):   # how the rates vary together, and why (covariance.py)
        sh('covariance.py', *MAIN, '--problem', 'cru_clean.npz', '--slip-block', SLIP_BLOCK, '--bins-block', BIN_BLOCK,
           *(['--ruptures', _rd] if _rd else []))           # into the runs' folder, results_dir()
        for _f in ('corr_faults.png', 'corr_bins.png'):
            if os.path.exists(os.path.join(results_dir(), _f)):
                show(os.path.join(results_dir(), _f))
else:
    print('fault-by-fault figures skipped: they need the rate vectors, written by the results cell above')

**The plate rate and the summed slip rates** (`plate_rate.txt`, `plate_rate_strips.csv`).

In [ ]:
vec = os.path.join(results_dir(), 'rate_vectors.npz')
_rd = rupture_dir()                                  # the faults' traces, rakes and dips, the ruptures' slips
if not _rd:
    print("plate-rate report skipped: it needs the model's OpenSHA rupture set (RUPTURE_SET)")
elif os.path.exists(vec):
    sh('plate_rate.py', _rd, '--run', results_dir(), '--problem', 'cru_clean.npz', '--slip-block', SLIP_BLOCK,
       '--cross-sections', CROSS_SECTIONS, '--slip-sum-l0', SLIP_SUM_L0,
       *(['--limits', PLATE_LIMITS] if PLATE_RATE else []))
else:
    print('plate-rate report skipped: it needs the rate vectors, written by the results cell above')

## 5. Rates for OpenSHA

The best fit's (`nnls/rates.csv`) and the posterior mean's (`spikeslab/rates.csv`) rates, one per rupture of the rupture set, with `solution_report.txt`; and the run's timings (`timings.json`) for later runs' plans.

In [ ]:
_rd = rupture_dir()                                  # looked for now: a solution folder that appeared during the run serves
for _vec, _col, _out, _what in (('nnls/nnls_vectors.npz', [], 'nnls', 'the best fit'),
                                (os.path.join(results_dir(), 'rate_vectors.npz'), ['--column', 'mean'], results_dir(),
                                 'the posterior mean')):
    clear_solution(_out)                             # an earlier run's would not match this one
    if not _rd:
        print(f'no OpenSHA rupture set (RUPTURE_SET): {_what} is not written as rates for OpenSHA')
    elif not os.path.exists(_vec):
        print(f'{_what} is not written as rates for OpenSHA: no rate vectors ({_vec})')
    else:                                            # a rate for every rupture of the rupture set, as in its own rates.csv
        sh('make_solution.py', _vec, *_col, '--ruptures', _rd, '--problem', 'cru_clean.npz', '--out', _out,
           '--slip-block', SLIP_BLOCK, '--bins-block', BIN_BLOCK)

write_timings()                                     # for later runs' plans (tools/run_plan.py)


## Open items

See `docs/REPORT.md`, "Open items".